# Mini-projet - Segmentation d'images (v2, dataset Kaggle, Google Colab)

**v2** de `mini_projet_segmentation.ipynb` : memes 7 missions, meme U-Net, meme Soft Dice
Loss, meme boucle d'entrainement - seule la source des donnees change (dataset Kaggle
Supervisely au lieu de Penn-Fudan telecharge en direct).

## Se connecter au T4

**`Select Kernel`** (en haut a droite) -> `Select Another Kernel...` -> `Colab` ->
connexion Google -> **`New Colab Server`** -> **`GPU`** -> **`T4`**

> Le GPU se choisit a la creation du serveur, nulle part ailleurs. Puis `Run All`.

## 📋 v2 vs v1 — tout ce qui change

| | v1 | v2 |
|---|---|---|
| Dataset | Penn-Fudan, 170 images, ~50 Mo | **Supervisely (Kaggle), 2667 images, ~4,3 Go** |
| Téléchargement | `urllib` + `zipfile` | **`kagglehub`** (1 ligne + cache) |
| Appariement image↔masque | tri parallèle (`sorted` ×2) | **par nom de fichier** (robuste) |
| Environnement visé | serveur du TP / PC local | **Google Colab (T4)** |

**Tout le reste est identique** : les 7 missions, le `TinyUNet`, la Soft Dice, la boucle.

> 📖 **Pour la théorie détaillée ligne par ligne** de ces parties communes (Dataset,
> U-Net, loss, boucle…), ouvre `mini_projet_segmentation_cours_complet.ipynb` — tout y
> est. Ici, on ne détaille que ce qui est **nouveau**.

La cellule ci-dessous est le garde Colab : `import google.colab` ne réussit **que** sur
Colab (le module n'existe pas ailleurs) → on installe `kagglehub` seulement là-bas.

In [ ]:
try:
    import google.colab
    DANS_COLAB = True
    !pip install -q kagglehub
except ImportError:
    DANS_COLAB = False
    print("Pas sur Colab : verifie que kagglehub est installe (pip install kagglehub).")

# Mini-projet — Segmentation d'images avec PyTorch

**Durée : 4 heures — en binôme**

## Sujet

Le fond flou de Zoom, les stickers Instagram, le détourage automatique de
Photoshop, remove.bg — tous ces outils reposent sur la **segmentation
d'images**. À partir d'une image, ils prédisent pour chaque pixel s'il
appartient à une personne ou au fond.

Dans ce TP, vous allez recoder ce genre de modèle de A à Z.

## Objectif

Entraîner un petit U-Net pour séparer les personnes du fond, en utilisant
PyTorch et le dataset public Penn-Fudan Pedestrian.

À la fin du TP, votre modèle doit être capable de prendre une image de votre
choix et de produire un masque binaire correct de la personne.

## Pipeline à construire

```
images + masques  →  Dataset PyTorch  →  U-Net  →  loss (BCE + Dice)
                     ↓
              boucle d'entraînement  →  évaluation  →  prédiction sur une nouvelle image
```

## Livrable final

À la fin du TP, chaque binôme présente **3 slides** en 5 minutes :
1. **Démarche** : dataset, split, architecture, hyperparamètres
2. **Résultats** : Dice / IoU sur le test set + courbes d'entraînement + 2 exemples
3. **Limites observées + 2 pistes d'amélioration**

## Règles du TP

1. Vous complétez uniquement les cellules avec **`MISSION`** et **`TODO`**.
   Les autres cellules doivent tourner telles quelles.
2. Les **tests automatiques** (assertions) doivent passer avant de continuer.
3. Vous êtes autorisés à consulter la documentation, googler, demander à
   l'encadrant, utiliser une IA pour **comprendre** un concept.
4. Vous n'êtes **PAS** autorisés à copier du code d'une IA sans savoir
   l'expliquer. L'encadrant passera vérifier.

## Planning suggéré (4 h)

| Section | Durée cible |
|---|---:|
| 0. Setup et téléchargement | 10 min |
| 1. MISSION 1 — Explorer le dataset | 20 min |
| 2. Split train/val/test | 5 min |
| 3. MISSION 2 — Dataset PyTorch | 45 min |
| 4. MISSION 3 — U-Net forward | 40 min |
| 5. MISSION 4 — Soft Dice Loss | 20 min |
| 6. MISSION 5 — Boucle d'entraînement | 30 min |
| 7. Entraînement + courbes | 20 min |
| 8. MISSION 6 — Choix du seuil | 15 min |
| 9. MISSION 7 — Prédiction nouvelle image | 15 min |
| 10. Bonus fond vert / flou | 10 min |
| 11. Bilan et préparation des slides | 30 min |
| **Total** | **~4 h** |

## Requirements

Bibliothèques nécessaires (déjà installées sur le serveur) :

| Bibliothèque | Utilité |
|---|---|
| `torch`, `torchvision` | Framework deep learning |
| `numpy` | Tableaux numériques |
| `matplotlib` | Visualisations |
| `Pillow` (PIL) | Lecture d'images |
| `opencv-python` (`cv2`) | Bonus : flou gaussien |

Si un import échoue, décommentez la cellule suivante pour installer.

In [ ]:
# Décommenter uniquement si un import échoue.
# !pip install torch torchvision numpy matplotlib pillow opencv-python

# 0 — Imports et configuration

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

import kagglehub

### Reproductibilité et hyperparamètres

On fixe une graine aléatoire pour que les résultats soient reproductibles.
Les hyperparamètres sont regroupés ici — modifiables facilement.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparametres - modifiables
IMG_SIZE = 128        # taille de redimensionnement (H = W)
BATCH_SIZE = 8        # nombre d'images par batch
EPOCHS = 5            # nombre d'epochs
BASE_CHANNELS = 16    # profondeur du premier bloc du U-Net
LEARNING_RATE = 1e-3  # taux d'apprentissage

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# 1 - Telechargement et exploration du dataset

**v2 : dataset Kaggle au lieu de Penn-Fudan.** On utilise
[Supervisely Filtered Segmentation Person Dataset](https://www.kaggle.com/datasets/tapakah68/supervisely-filtered-segmentation-person-dataset) :
**2667 images** de personnes en pied, avec leurs masques de segmentation binaires.

> ATTENTION taille : ce dataset fait **~4,3 Go**. Sur Colab ce n'est pas un probleme
> (le disque de la VM est jetable et largement suffisant), mais evite de le telecharger
> en local sur ton PC.

Le dataset est organise en 2 dossiers avec les memes noms de fichiers :
```
.../images/xxx.jpg   (ou .png)
.../masks/xxx.png
```
`kagglehub.dataset_download(...)` le telecharge et le decompresse tout seul, et renvoie
le chemin local. On va chercher `images/` et `masks/` avec `rglob` pour ne pas dependre
du nom exact du dossier parent.

## 🧠 Le cours — `kagglehub` et l'appariement par NOM

### kagglehub en 3 points

1. `dataset_download("auteur/nom-du-dataset")` télécharge, décompresse, et renvoie le
   **chemin local** — 1 ligne remplace tout le bloc `urllib` + `zipfile` de la v1
2. Il gère un **cache** (`~/.cache/kagglehub`) : relancer la cellule ne retélécharge rien
3. Sur Colab, le disque de la VM est jetable : les 4,3 Go ne coûtent rien

### ⚠️ Le changement le plus important vs la v1 : l'appariement par NOM

La v1 appariait par **tri parallèle** : `sorted(images)` et `sorted(masks)`, puis
`zip` — ça marche parce que Penn-Fudan garantit un masque par image, dans le même ordre.

Ici on ne peut plus : les extensions **diffèrent** (`xxx.jpg` ↔ `xxx.png`) et rien ne
garantit qu'aucun fichier ne manque. Donc on apparie **par nom de fichier** :
`find_mask_for` cherche dans `masks/` un fichier du **même nom** (`.stem`), extension
libre. Une image sans masque est **ignorée proprement** — pas de crash, pas de décalage.

> C'est la version robuste du piège du `sorted()` de la v1 : au lieu d'espérer que deux
> listes triées s'alignent, on **vérifie** la correspondance nom par nom.

In [ ]:
# Telechargement via kagglehub (~4,3 Go, patience la premiere fois)
# kagglehub gere un CACHE (~/.cache/kagglehub) : le 2e run ne retelecharge RIEN
dataset_path = kagglehub.dataset_download("tapakah68/supervisely-filtered-segmentation-person-dataset")
DATA_ROOT = Path(dataset_path)                    # le chemin local ou tout a ete decompresse
print("Dataset dans :", DATA_ROOT)

# rglob = glob RECURSIF : trouve images/ et masks/ OU QU'ILS SOIENT dans l'arborescence
# (on ne depend pas du nom exact du dossier parent — lecon du zip imbrique de la v1 !)
images_dir = next(DATA_ROOT.rglob("images"))      # next() prend le 1er resultat du generateur
masks_dir = next(DATA_ROOT.rglob("masks"))


def find_mask_for(image_path, masks_dir):
    """Le masque partage le NOM de l'image (extension parfois differente : .jpg vs .png).
    .stem = le nom sans extension ; glob(stem + '.*') = ce nom, n'importe quelle extension."""
    candidates = list(masks_dir.glob(image_path.stem + ".*"))
    return candidates[0] if candidates else None   # None si aucun masque trouve


image_paths = []
mask_paths = []
for img in sorted(images_dir.glob("*")):           # sorted : ordre stable et reproductible
    if img.suffix.lower() not in (".jpg", ".jpeg", ".png"):
        continue                                   # ignore les fichiers parasites (readme, .txt...)
    m = find_mask_for(img, masks_dir)              # cherche le masque DU MEME NOM
    if m is not None:                              # une image sans masque est IGNOREE (pas un crash)
        image_paths.append(img)
        mask_paths.append(m)

# comparaison chainee : autant d'images que de masques, ET au moins 1
assert len(image_paths) == len(mask_paths) > 0
print(f"\n{len(image_paths)} couples image/masque disponibles")

## MISSION 1 — Lire une image et binariser un masque

Complétez la cellule suivante pour :

1. Ouvrir la première image (`image_paths[0]`) en mode **RGB**
2. Ouvrir son masque correspondant (`mask_paths[0]`) en mode **niveaux de gris** (`"L"`)
3. Convertir le masque en tableau **NumPy**
4. Créer un masque **binaire** de type `float32` (fond = 0.0, personne = 1.0)
5. Calculer le pourcentage de pixels appartenant à une personne

**Documentation utile** :
- `PIL.Image.open(path).convert("RGB")` et `.convert("L")`
- `np.array(pil_image)` transforme une image PIL en tableau NumPy
- Une comparaison comme `array > 0` produit un tableau booléen
- `.astype(np.float32)` convertit en float
- `.mean()` sur un tableau de 0 et 1 donne la proportion de 1

In [ ]:
# --- MISSION 1 ---

image = Image.open(image_paths[0]).convert("RGB")
mask_raw = Image.open(mask_paths[0]).convert("L")

if image is None or mask_raw is None:
    raise NotImplementedError("Complétez le chargement de l'image et du masque")

mask_array = np.array(mask_raw)
mask_binary = (mask_array > 0).astype(np.float32)

if mask_array is None or mask_binary is None:
    raise NotImplementedError("Complétez la binarisation")

foreground_ratio = mask_binary.mean()

if foreground_ratio is None:
    raise NotImplementedError("Complétez le calcul du pourcentage")

# --- Tests automatiques ---
assert image.mode == "RGB", "L'image doit être en RGB"
assert mask_binary.ndim == 2, "Le masque binaire doit avoir 2 dimensions"
assert mask_binary.dtype == np.float32, "Le masque binaire doit être en float32"
assert set(np.unique(mask_binary)).issubset({0.0, 1.0}), "Le masque doit contenir uniquement 0 et 1"
assert 0.0 < foreground_ratio < 1.0

print(f"Forme image  : {np.array(image).shape}")
print(f"Forme masque : {mask_binary.shape}")
print(f"Pixels personne : {100 * foreground_ratio:.1f} %")

# Affichage
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(image); axes[0].set_title("Image RGB")
axes[1].imshow(mask_raw, cmap="nipy_spectral"); axes[1].set_title("Masque brut (multi-classes)")
axes[2].imshow(mask_binary, cmap="gray"); axes[2].set_title("Masque binaire")
for ax in axes:
    ax.axis("off")
plt.show()

print("\nMISSION 1 validée")

# 2 — Séparation train / validation / test

On coupe le dataset en 3 parties :
- **Train (70 %)** : ce sur quoi le modèle apprend
- **Validation (15 %)** : pour choisir le meilleur checkpoint
- **Test (15 %)** : réservé pour l'évaluation finale — on n'y touche pas avant

Le split est **reproductible** grâce à la graine aléatoire.

In [ ]:
pairs = list(zip(image_paths, mask_paths))
random.Random(SEED).shuffle(pairs)

n_total = len(pairs)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

train_pairs = pairs[:n_train]
val_pairs = pairs[n_train:n_train + n_val]
test_pairs = pairs[n_train + n_val:]

print(f"Train      : {len(train_pairs)}")
print(f"Validation : {len(val_pairs)}")
print(f"Test       : {len(test_pairs)}")

# Vérification : les splits ne se chevauchent pas
assert set(train_pairs).isdisjoint(val_pairs)
assert set(train_pairs).isdisjoint(test_pairs)
assert set(val_pairs).isdisjoint(test_pairs)

# 3 — Construire le Dataset PyTorch

Un `Dataset` PyTorch encapsule la logique de chargement d'un élément. Il doit
implémenter deux méthodes obligatoires : `__len__` et `__getitem__`.

Pour ce projet, chaque élément doit retourner un **tuple** `(image, mask)` où :
- `image` est un tenseur de forme `(3, H, W)` en `float32`
- `mask` est un tenseur de forme `(1, H, W)` avec des valeurs 0.0 ou 1.0

**Points d'attention** :
- L'image et le masque doivent subir **la même transformation géométrique**
  (même flip, même resize) sinon l'annotation ne correspond plus.
- Pour redimensionner un **masque**, on utilise l'interpolation `NEAREST`
  (plus proche voisin) pour ne pas créer de valeurs intermédiaires (0.5, 0.7...).
- Pour redimensionner une **image**, on utilise l'interpolation `BILINEAR`.

## MISSION 2 — Compléter la classe PersonDataset

**Documentation utile** :
- `PIL.Image.open(path).convert("RGB")` et `.convert("L")`
- `torchvision.transforms.functional.to_tensor(image)` (importé comme `TF.to_tensor`)
  → convertit une image PIL en tenseur `float32` normalisé entre 0 et 1
- `torch.from_numpy(array)` → convertit un tableau NumPy en tenseur
- `.unsqueeze(0)` → ajoute une dimension en position 0 (utile pour passer de (H,W) à (1,H,W))

In [ ]:
# --- MISSION 2 ---

class PersonDataset(Dataset):
    def __init__(self, pairs, size=128, augment=False):
        self.pairs = list(pairs)
        self.size = size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        if image is None or mask is None:
            raise NotImplementedError("Complétez le chargement image/masque")

        # Redimensionnement (déjà fait pour vous)
        image = TF.resize(
            image, [self.size, self.size],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True,
        )
        mask = TF.resize(
            mask, [self.size, self.size],
            interpolation=InterpolationMode.NEAREST,
        )

        # Data augmentation : flip horizontal aléatoire (déjà fait)
        # Attention : on applique le MÊME flip à l'image ET au masque
        if self.augment and random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        image_tensor = TF.to_tensor(image)

        # Pour le masque : convertir en NumPy, binariser, en float32
        mask_np = (np.array(mask) > 0).astype(np.float32)
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0)

        if image_tensor is None or mask_tensor is None:
            raise NotImplementedError("Complétez la conversion en tenseurs")

        return image_tensor, mask_tensor


# --- Tests automatiques ---
debug_dataset = PersonDataset(train_pairs[:3], size=IMG_SIZE, augment=False)

assert len(debug_dataset) == 3
img_test, mask_test = debug_dataset[0]

assert img_test.shape == (3, IMG_SIZE, IMG_SIZE), f"Shape image attendue (3, {IMG_SIZE}, {IMG_SIZE}), obtenue {img_test.shape}"
assert mask_test.shape == (1, IMG_SIZE, IMG_SIZE), f"Shape masque attendue (1, {IMG_SIZE}, {IMG_SIZE}), obtenue {mask_test.shape}"
assert img_test.dtype == torch.float32
assert mask_test.dtype == torch.float32
assert set(torch.unique(mask_test).tolist()).issubset({0.0, 1.0})

print("MISSION 2 validée")
print(f"Image  : shape {img_test.shape}, dtype {img_test.dtype}")
print(f"Masque : shape {mask_test.shape}, dtype {mask_test.dtype}")

### Construction des DataLoaders

In [ ]:
train_dataset = PersonDataset(train_pairs, IMG_SIZE, augment=True)
val_dataset = PersonDataset(val_pairs, IMG_SIZE, augment=False)
test_dataset = PersonDataset(test_pairs, IMG_SIZE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=0, pin_memory=True)

images_batch, masks_batch = next(iter(train_loader))
print(f"Batch images  : {images_batch.shape}")
print(f"Batch masques : {masks_batch.shape}")

### Vérification visuelle — les masques doivent bien correspondre aux personnes

In [ ]:
images_batch, masks_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(images_batch[i].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title(f"Image #{i}")
    axes[0, i].axis("off")
    axes[1, i].imshow(masks_batch[i, 0], cmap="gray")
    axes[1, i].set_title(f"Masque #{i}")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

# 4 — Construire un petit U-Net

Le U-Net est composé de :
- Un **encoder** qui compresse l'information spatiale (downsample par max-pooling)
- Un **bottleneck** au fond du U
- Un **decoder** qui reconstruit la carte à la résolution originale (upsample)
- Des **skip connections** qui transportent l'information spatiale de l'encoder vers le decoder

Les skip connections sont **essentielles** : elles permettent au decoder de
récupérer les détails fins perdus lors du downsampling.

## Structure de notre TinyUNet

```
Input (3, 128, 128)
    │
    ▼
enc1 → skip1 ──────────────────────────────┐
    │                                       │
    ▼ pool                                  │
enc2 → skip2 ────────────────────┐          │
    │                             │          │
    ▼ pool                        │          │
bottleneck                        │          │
    │                             │          │
    ▼ up                          │          │
concat + dec2  ←──────────────────┘          │
    │                                        │
    ▼ up                                     │
concat + dec1  ←─────────────────────────────┘
    │
    ▼
head → Output (1, 128, 128)   ← logits, PAS de sigmoid ici
```

Le bloc `DoubleConv` est déjà fourni : deux convolutions 3×3 avec BatchNorm et ReLU.

In [ ]:
class DoubleConv(nn.Module):
    """Bloc standard du U-Net : (Conv → BN → ReLU) × 2"""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

## MISSION 3 — Compléter le forward du TinyUNet

Vous devez :
1. Passer `x` dans `enc1` pour obtenir `skip1`
2. Downsampler avec `pool1`, puis passer dans `enc2` pour obtenir `skip2`
3. Downsampler avec `pool2`, puis passer dans `bottleneck`
4. Upsampler avec `up2`, **concaténer avec `skip2`** sur la dimension des canaux, puis `dec2`
5. Upsampler avec `up1`, **concaténer avec `skip1`** sur la dimension des canaux, puis `dec1`
6. Passer dans `head` pour obtenir la sortie finale

**Documentation utile** :
- `torch.cat([a, b], dim=1)` concatène deux tenseurs sur l'axe des canaux
- Le format des tenseurs PyTorch est `(batch, channels, H, W)` — la dim 1 est celle des canaux

In [ ]:
# --- MISSION 3 ---

class TinyUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(3, base)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base * 2, base * 4)

        # Decoder
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)   # entrée = up + skip → base*4

        self.up1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)       # entrée = up + skip → base*2

        # Head : produit 1 canal de sortie (segmentation binaire)
        self.head = nn.Conv2d(base, 1, kernel_size=1)

    def forward(self, x):
        # --- Encoder ---
        skip1 = self.enc1(x)
        skip2 = self.enc2(self.pool1(skip1))
        x = self.bottleneck(self.pool2(skip2))

        if skip1 is None or skip2 is None or x is None:
            raise NotImplementedError("Complétez l'encoder et le bottleneck")

        # --- Decoder niveau 2 ---
        x = self.up2(x)
        x = torch.cat([x, skip2], dim=1)

        if x is None:
            raise NotImplementedError("Complétez la première skip connection")

        x = self.dec2(x)

        # --- Decoder niveau 1 ---
        x = self.up1(x)
        x = torch.cat([x, skip1], dim=1)

        if x is None:
            raise NotImplementedError("Complétez la seconde skip connection")

        x = self.dec1(x)
        return self.head(x)


# --- Tests automatiques ---
model = TinyUNet(BASE_CHANNELS).to(DEVICE)

dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
dummy_output = model(dummy_input)

assert dummy_output.shape == (2, 1, IMG_SIZE, IMG_SIZE), \
    f"Shape sortie attendue (2, 1, {IMG_SIZE}, {IMG_SIZE}), obtenue {dummy_output.shape}"

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("MISSION 3 validée")
print(f"Paramètres entraînables : {n_params:,}")
print(f"Input shape  : {dummy_input.shape}")
print(f"Output shape : {dummy_output.shape}")

# 5 — Loss et métriques

En segmentation binaire, on combine généralement :
- **BCE** (Binary Cross-Entropy) : mesure l'erreur pixel par pixel
- **Dice loss** : mesure le recouvrement au niveau régional (bon pour les classes déséquilibrées)

Pour l'évaluation finale, on utilise deux métriques :
- **Dice score** : `2 × |A ∩ B| / (|A| + |B|)`
- **IoU (Jaccard)** : `|A ∩ B| / |A ∪ B|`

Les deux valent 1 pour une prédiction parfaite, 0 pour une prédiction totalement fausse.

**Note importante** : la loss travaille sur des **probabilités continues**
(sortie de sigmoid), tandis que le Dice score final travaille sur des
**prédictions binaires** (après seuillage). C'est parce que le seuillage
n'est pas différentiable — on ne peut pas l'utiliser dans la loss.

## MISSION 4 — Implémenter la Soft Dice Loss

La Soft Dice utilise directement les probabilités (sortie de sigmoid) au lieu
de valeurs binaires. Cela permet le calcul du gradient.

Formule (moyenne par image) :

```
soft_dice = (2 × Σ(p × t) + ε) / (Σ(p) + Σ(t) + ε)
loss = 1 - soft_dice.mean()
```

où `p` = probabilités (sigmoid des logits), `t` = target binaire, `ε` = petit epsilon
pour éviter la division par zéro.

**Documentation utile** :
- `torch.sigmoid(logits)` → applique la sigmoïde élément par élément
- `.sum(dim=dims)` avec `dims=(1,2,3)` somme sur canaux, H et W (garde le batch)
- `.mean()` moyenne finale sur le batch

In [ ]:
# --- MISSION 4 ---

BCE = nn.BCEWithLogitsLoss()


def soft_dice_loss(logits, targets, eps=1e-6):
    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)   # sommer sur canaux, H, W (garder la dim batch)

    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)
    dice = (2 * intersection + eps) / (denominator + eps)

    if intersection is None or denominator is None or dice is None:
        raise NotImplementedError("Complétez la formule du Soft Dice")

    return 1 - dice.mean()


def combined_loss(logits, targets):
    """Combinaison classique : moitié BCE, moitié Dice."""
    return 0.5 * BCE(logits, targets) + 0.5 * soft_dice_loss(logits, targets)


@torch.no_grad()
def binary_metrics(logits, targets, threshold=0.5, eps=1e-6):
    """Calcule Dice et IoU par image du batch."""
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    dims = (1, 2, 3)
    intersection = (predictions * targets).sum(dim=dims)
    pred_area = predictions.sum(dim=dims)
    target_area = targets.sum(dim=dims)
    union = pred_area + target_area - intersection

    dice = (2 * intersection + eps) / (pred_area + target_area + eps)
    iou = (intersection + eps) / (union + eps)
    return dice, iou


# --- Tests automatiques ---
# Cas parfait : prédictions égales à la target
perfect_targets = torch.tensor([[[[1., 0.], [0., 1.]]]], device=DEVICE)
perfect_logits = torch.tensor([[[[10., -10.], [-10., 10.]]]], device=DEVICE)
loss_perfect = soft_dice_loss(perfect_logits, perfect_targets).item()

# Cas totalement faux : prédictions inversées
bad_logits = -perfect_logits
loss_bad = soft_dice_loss(bad_logits, perfect_targets).item()

assert loss_perfect < 0.01, f"Loss cas parfait attendue < 0.01, obtenue {loss_perfect}"
assert loss_bad > 0.90, f"Loss cas faux attendue > 0.90, obtenue {loss_bad}"

print("MISSION 4 validée")
print(f"Loss (prédiction parfaite) : {loss_perfect:.4f}")
print(f"Loss (prédiction inversée) : {loss_bad:.4f}")

# 6 — Boucle d'entraînement

La boucle d'entraînement standard PyTorch se déroule ainsi pour chaque batch :

1. Mettre le modèle en mode entraînement (`model.train()`) — une seule fois par epoch, avant la boucle
2. Déplacer les tenseurs vers le GPU (`.to(device)`)
3. **Zéro les gradients** de l'optimiseur (`optimizer.zero_grad()`)
4. **Forward** : passer l'input dans le modèle
5. Calculer la **loss**
6. **Backward** : calculer les gradients (`loss.backward()`)
7. **Step** : mettre à jour les poids (`optimizer.step()`)

Cet ordre est **impératif** — inverser l'un des trois derniers casse tout.

## MISSION 5 — Compléter la fonction train_one_epoch

**Documentation utile** :
- `model.train()` et `model.eval()`
- `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`
- `.item()` pour extraire une valeur scalaire d'un tenseur

In [ ]:
# --- MISSION 5 ---

def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    all_dice = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = combined_loss(logits, masks)

        if logits is None or loss is None:
            raise NotImplementedError("Complétez forward et loss")

        loss.backward()
        optimizer.step()

        dice, _ = binary_metrics(logits.detach(), masks)
        total_loss += loss.item() * images.size(0)
        all_dice.extend(dice.cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "dice": float(np.mean(all_dice)),
    }


@torch.no_grad()
def evaluate(model, loader, device, threshold=0.5):
    """Évaluation sur un loader (val ou test). Retourne loss, dice, iou moyens."""
    model.eval()

    total_loss = 0.0
    all_dice = []
    all_iou = []

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = combined_loss(logits, masks)
        dice, iou = binary_metrics(logits, masks, threshold)

        total_loss += loss.item() * images.size(0)
        all_dice.extend(dice.cpu().tolist())
        all_iou.extend(iou.cpu().tolist())

    return {
        "loss": total_loss / len(loader.dataset),
        "dice": float(np.mean(all_dice)),
        "iou": float(np.mean(all_iou)),
    }

### Lancement de l'entraînement

Sur GPU, chaque epoch prend quelques secondes. Sur CPU, comptez ~1 min par epoch.
On sauvegarde le **meilleur modèle** (meilleur Dice sur la validation) pour
l'évaluation finale.

In [ ]:
# Réinitialiser le modèle et l'optimiseur pour un entraînement propre
model = TinyUNet(BASE_CHANNELS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

history = {"train_loss": [], "train_dice": [], "val_loss": [], "val_dice": []}
best_val_dice = -1.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_stats = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_stats = evaluate(model, val_loader, DEVICE)

    history["train_loss"].append(train_stats["loss"])
    history["train_dice"].append(train_stats["dice"])
    history["val_loss"].append(val_stats["loss"])
    history["val_dice"].append(val_stats["dice"])

    if val_stats["dice"] > best_val_dice:
        best_val_dice = val_stats["dice"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        marker = "  ← meilleur"
    else:
        marker = ""

    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss={train_stats['loss']:.3f}  dice={train_stats['dice']:.3f} | "
          f"val loss={val_stats['loss']:.3f}  dice={val_stats['dice']:.3f}{marker}")

# Recharger le meilleur checkpoint dans le modèle
assert best_state is not None
model.load_state_dict(best_state)
print(f"\nMeilleur Dice de validation : {best_val_dice:.4f}")

### Courbes d'entraînement

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs_range, history["val_loss"], "s-", label="Validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, history["train_dice"], "o-", label="Train")
axes[1].plot(epochs_range, history["val_dice"], "s-", label="Validation")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Dice")
axes[1].set_title("Dice score"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# 7 — Évaluation finale et choix du seuil

Le seuil de binarisation par défaut est `0.5`, mais ce n'est pas toujours optimal.
On cherche le meilleur seuil sur la **validation**, puis on l'utilise sur le test.

**Attention** : on ne cherche PAS le meilleur seuil sur le test set — ce serait tricher.

## MISSION 6 — Chercher le meilleur seuil sur la validation

Complétez la cellule pour :
1. Évaluer le modèle sur `val_loader` avec chaque seuil de `thresholds`
2. Récupérer le Dice de validation pour chaque seuil
3. Sélectionner le seuil qui donne le meilleur Dice

**Documentation utile** : la fonction `evaluate(model, loader, device, threshold=...)`
prend un argument `threshold`.

In [ ]:
# --- MISSION 6 ---

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]
validation_scores = []

for threshold in thresholds:
    stats = evaluate(model, val_loader, DEVICE, threshold=threshold)

    if stats is None:
        raise NotImplementedError("Évaluez le modèle pour chaque seuil")

    validation_scores.append(stats["dice"])
    print(f"Seuil {threshold:.2f} → Dice validation = {stats['dice']:.4f}")

best_index = int(np.argmax(validation_scores))
best_threshold = thresholds[best_index]

if best_index is None or best_threshold is None:
    raise NotImplementedError("Sélectionnez le meilleur seuil")

assert best_threshold in thresholds
print(f"\nMeilleur seuil : {best_threshold:.2f}")

### Évaluation finale sur le test set

In [ ]:
test_stats = evaluate(model, test_loader, DEVICE, threshold=best_threshold)

print(f"=== Résultats finaux sur le test set ===")
print(f"Loss  : {test_stats['loss']:.4f}")
print(f"Dice  : {test_stats['dice']:.4f}")
print(f"IoU   : {test_stats['iou']:.4f}")

# 8 — Visualisation des prédictions

Pour chaque image : image originale, vérité terrain, probabilité prédite
(avant seuillage), masque final (après seuillage).

Regardez attentivement les erreurs — c'est ce qui vous permettra de proposer
des pistes d'amélioration dans votre restitution finale.

In [ ]:
@torch.no_grad()
def show_predictions(model, loader, threshold, n=4):
    model.eval()
    images, masks = next(iter(loader))

    logits = model(images.to(DEVICE))
    probabilities = torch.sigmoid(logits).cpu()
    predictions = (probabilities >= threshold).float()

    n = min(n, len(images))
    fig, axes = plt.subplots(n, 4, figsize=(14, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)

    for i in range(n):
        axes[i, 0].imshow(images[i].permute(1, 2, 0).clamp(0, 1))
        axes[i, 1].imshow(masks[i, 0], cmap="gray")
        axes[i, 2].imshow(probabilities[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[i, 3].imshow(predictions[i, 0], cmap="gray")

        for j in range(4):
            axes[i, j].axis("off")

    axes[0, 0].set_title("Image")
    axes[0, 1].set_title("Vérité terrain")
    axes[0, 2].set_title("Probabilité")
    axes[0, 3].set_title("Prédiction (seuillée)")

    plt.tight_layout()
    plt.show()


show_predictions(model, test_loader, best_threshold, n=4)

## MISSION 7 — Tester le modèle sur une nouvelle image

Pour tester votre modèle sur une image qui n'est pas dans le dataset :

1. Créez un dossier `photos_test/` à côté du notebook (la cellule le fait pour vous)
2. Uploadez-y une image contenant une ou plusieurs personnes (glisser-déposer dans VS Code)
3. Complétez le code ci-dessous pour obtenir la probabilité et la prédiction

**Documentation utile** :
- `torch.sigmoid(logits)` pour convertir les logits en probabilités
- `.squeeze()` pour retirer les dimensions de taille 1
- `.cpu().numpy()` pour convertir un tenseur GPU en tableau NumPy affichable

In [ ]:
# --- MISSION 7 ---

# Placez une image dans ce dossier avant d'exécuter la cellule
PHOTOS_DIR = Path("../photos_test")
PHOTOS_DIR.mkdir(exist_ok=True)

photos = list(PHOTOS_DIR.glob("*.jpg")) + list(PHOTOS_DIR.glob("*.png"))
if not photos:
    raise FileNotFoundError(
        f"Aucune image trouvée dans {PHOTOS_DIR}. "
        f"Ajoutez-y une .jpg ou .png contenant une personne."
    )

photo_path = photos[0]
print(f"Image testée : {photo_path.name}")

# Charger et prétraiter
new_image = Image.open(photo_path).convert("RGB")
resized = TF.resize(new_image, [IMG_SIZE, IMG_SIZE],
                    interpolation=InterpolationMode.BILINEAR, antialias=True)
input_tensor = TF.to_tensor(resized).unsqueeze(0).to(DEVICE)

# Inference
model.eval()
with torch.no_grad():
    logits = model(input_tensor)

probability = torch.sigmoid(logits).squeeze().cpu().numpy()
prediction = (probability >= best_threshold).astype(np.float32)

if probability is None or prediction is None:
    raise NotImplementedError("Complétez la conversion en probabilités et le seuillage")

# --- Tests automatiques ---
assert probability.shape == (IMG_SIZE, IMG_SIZE)
assert prediction.shape == (IMG_SIZE, IMG_SIZE)

# Affichage
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(new_image); axes[0].set_title("Photo originale"); axes[0].axis("off")
axes[1].imshow(probability, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Probabilité prédite"); axes[1].axis("off")
axes[2].imshow(prediction, cmap="gray")
axes[2].set_title(f"Masque (seuil {best_threshold:.2f})"); axes[2].axis("off")
plt.tight_layout()
plt.show()

print("\nMISSION 7 validée")

## Bonus — Remplacer le fond de votre image

Maintenant que vous avez le masque, vous pouvez recréer l'effet Zoom :
remplacer le fond par une couleur unie ou par un flou gaussien.

Cette cellule utilise `cv2` (OpenCV) pour le flou gaussien.

In [ ]:
import cv2

# Redimensionner le masque à la taille originale de l'image
original_size = new_image.size   # (W, H) pour PIL
prediction_pil = Image.fromarray((prediction * 255).astype(np.uint8))
prediction_full = np.array(prediction_pil.resize(original_size, Image.NEAREST)) > 127

image_np = np.array(new_image)

# Version fond vert
result_green = image_np.copy()
result_green[~prediction_full] = [0, 255, 0]

# Version fond flou (style Zoom)
image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)
blurred = cv2.GaussianBlur(image_bgr, (45, 45), 0)
result_blur = image_bgr.copy()
result_blur[~prediction_full] = blurred[~prediction_full]
result_blur = cv2.cvtColor(result_blur, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(image_np); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(result_green); axes[1].set_title("Fond vert"); axes[1].axis("off")
axes[2].imshow(result_blur); axes[2].set_title("Fond flou (style Zoom)"); axes[2].axis("off")
plt.tight_layout()
plt.show()

# Bilan du projet

Complétez cette grille avec vos résultats pour la présentation finale :

| Élément | Valeur |
|---|---|
| Meilleur seuil trouvé | ... |
| Dice sur test set | ... |
| IoU sur test set | ... |
| Nombre de paramètres du modèle | ... |
| Nombre d'epochs d'entraînement | ... |
| Temps total d'entraînement | ... |

### Analyse d'erreurs
Trouvez dans le test set :
- **1 image où le modèle marche très bien** — sauvegardez-la pour vos slides
- **1 image où le modèle échoue** — sauvegardez-la et notez pourquoi
  (contour flou, personne petite, occlusion, éclairage difficile...)

### Deux pistes d'amélioration à proposer
Regardez la documentation de `torchvision.transforms` ou explorez d'autres
architectures pour identifier deux améliorations concrètes à tester
(plus de data augmentation, encoder pré-entraîné, plus d'epochs, resize
différent, dataset plus grand...).

---

## Rendu final — 3 slides

1. **Démarche** : dataset, split, architecture, hyperparamètres
2. **Résultats** : Dice / IoU test + courbes + 2 exemples
3. **Limites + 2 pistes d'amélioration**

Restitution : **5 minutes par binôme**.

---

**Bravo, vous avez fait un projet complet de segmentation en 4h.**